In [38]:
# conecta o colab ao google drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
# importa as bibliotecas necessárias
import os
import pandas as pd
import re
import sqlite3

In [40]:
# caminho base para acesso as pastas
url_base = "/content/drive/MyDrive/teste_vena_analista_dados/"

### Unidade

In [41]:
# lê o arquivo de unidade de negócio
dim_unidade = pd.read_excel(url_base + "/Arquivos Fontes/Arquivos de Apoio/Unidades e Estrutura.xlsx", sheet_name="Unidades")

In [42]:
# renomeia as colunas
dim_unidade = dim_unidade.rename(columns={
  "Nome no Excel": "NomeNaFonte",
  "Codigo Unidade": "CodigoUnidade",
  "Tipo Unidade": "TipoUnidade",
  "Nome para Relatorios": "NomeRelatorios",
})

In [43]:
# cria a coluna de id
dim_unidade["IdUnidade"] = range(1, len(dim_unidade) + 1)

In [44]:
dim_unidade.head(10)

,NomeNaFonte,CodigoUnidade,TipoUnidade,NomeRelatorios,IdUnidade
0,9999111 - ESCRITÓRIO,9999111,Escritório,Escritório - 9999111,1
1,Unidade 1 - 11111,11111,Varejo,Unidade 1 - 11111,2
2,Unidade 2 - 11112 - Atacado,11112,Atacado,Unidade 2 - 11112,3
3,Unidade 3 - 11113,11113,Varejo,Unidade 3 - 11113,4
4,Unidade 4 - 11114 - Atacado,11114,Atacado,Unidade 4 - 11114,5
5,Unidade 5 - 11115 - Atacado,11115,Atacado,Unidade 5 - 11115,6
6,Unidade 6 - 11116,11116,Varejo,Unidade 6 - 11116,7
7,Unidade 7 - 11117,11117,Varejo,Unidade 7 - 11117,8


In [45]:
# exporta a transformação
dim_unidade.to_csv(url_base + "dim_unidade.csv", index=False)

### Estrutura

In [46]:
# lê o arquivo de estrutura da dre
dim_estrutura = pd.read_excel(url_base + "Arquivos Fontes/Arquivos de Apoio/Unidades e Estrutura.xlsx", sheet_name="EstruturaDRE")

In [47]:
# substitui o valor ",0" por nada ("" = string vazia)
dim_estrutura[["IdEstrutura_sukey", "IdEstrutura_sukey_pai"]] = dim_estrutura[["IdEstrutura_sukey", "IdEstrutura_sukey_pai"]].astype(str).apply(lambda col: col.str.replace(",0", ""))

In [48]:
# converte colunas para inteiro, sem preencher nulos
for col in ["Int1", "Int2", "Int3", "Int4", "SomaDe"]:
  if col in dim_estrutura.columns:
    dim_estrutura[col] = pd.to_numeric(dim_estrutura[col], errors="coerce")
    dim_estrutura[col] = dim_estrutura[col].astype("Int64")

In [49]:
# cria a coluna de id
dim_estrutura["IdEstrutura"] = range(1, len(dim_estrutura) + 1)

In [50]:
dim_estrutura.head()

,IdEstrutura_sukey,LinhaEstrutura,LinhaRelatorio,TipoLinha,GrupoDaConta,TipoEstrutura,Int1,Int2,Int3,Int4,Subgrupo,Subgrupo2,IdEstrutura_sukey_pai,SomaDe,IdEstrutura
0,1,1 - Receita Operacional Bruta,Receita Operacional Bruta,Grupo,(+) Receita Operacional Bruta,Receita,1,<NA>,<NA>,<NA>,NaN,NaN,nan,3,1
1,1.1,1.1 - Vendas de mercadorias,Vendas de mercadorias,Conta,(+) Receita Operacional Bruta,Receita,1,1,<NA>,<NA>,1.1 - Vendas de mercadorias,1.1 - Vendas de mercadorias,1,<NA>,2
2,1.2,1.2 - Prestação de Serviços,Prestação de Serviços,Conta,(+) Receita Operacional Bruta,Receita,1,2,<NA>,<NA>,1.2 - Prestação de Serviços,1.2 - Prestação de Serviços,1,<NA>,3
3,2,2 - Deduções da Receita Operacional Bruta,Deduções da Receita Operacional Bruta,Grupo,(-) Deduções Da Receita Operacional Bruta,Despesa,2,<NA>,<NA>,<NA>,NaN,NaN,nan,3,4
4,2.1,2.1 - Impostos Incidentes sobre Vendas,Impostos Incidentes sobre Vendas,Subgrupo,(-) Deduções Da Receita Operacional Bruta,Despesa,2,1,<NA>,<NA>,2.1 - Impostos Incidentes sobre Vendas,NaN,2,<NA>,5


In [51]:
# exporta a transformação
dim_estrutura.to_csv(url_base + "dim_estrutura.csv", index=False)

### Staging

In [52]:
# cria um dataframe vazio
dfs = []

# lê os arquivos mensais, extraindo a data e adicionando em nova coluna
for nome_arquivo in os.listdir(url_base + "Arquivos Fontes/Base Analitica/"):

  if nome_arquivo.endswith(".xlsx"):

    # extrai a data no nome do arquivo para preencher na coluna
    match = re.search(r"(\d{2})-(\d{4})", nome_arquivo)

    if match:
      # coleta mês, ano e adiciona dia
      mes = match.group(1)
      ano = match.group(2)
      data = f"01-{mes}-{ano}"

      # armazena os dataframes numa variável
      df = pd.read_excel(os.path.join(url_base, "Arquivos Fontes/Base Analitica", nome_arquivo), sheet_name="DRE")

      # adiciona a coluna com a data extraída
      df["MesRef"] = data
      dfs.append(df)

In [53]:
# junta os arquivos em dataframe único
staging_dre = pd.concat(dfs, ignore_index=True)

In [54]:
# pivota o dataframe
staging_dre = staging_dre.melt(id_vars=["MesRef", "Nome"], var_name="Unidade", value_name="Valor")

In [55]:
# remove as linhas em que a Unidade é "Total"
staging_dre = staging_dre[staging_dre["Unidade"] != "Total"]

In [56]:
# remove espaços vazios das colunas
staging_dre[["Nome", "Unidade"]] = staging_dre[["Nome", "Unidade"]].apply(lambda x: x.str.strip())

In [57]:
# renomeia a coluna
staging_dre = staging_dre.rename(columns={"Nome": "LinhaEstrutura"})

In [58]:
# converte e ajusta o campo de data
staging_dre['MesRef'] = pd.to_datetime(staging_dre['MesRef'], format='%d-%m-%Y')
staging_dre['MesRef'] = staging_dre['MesRef'].dt.strftime('%Y-%m-%d')

In [59]:
# verifica os valores da coluna "LinhaEstrutura" para identificar diferenças deles nos dataframes
merged = pd.merge(dim_estrutura, staging_dre, on="LinhaEstrutura", how="outer", indicator=True)
diferenca = merged[merged["_merge"] != "both"]
list(diferenca["LinhaEstrutura"].unique())

['11 - Resultado Operacional -Ebitda',
 '11 - Resultado Operacional Ebitda',
 '12 - (+/-) Resultado Financeiro Líquido',
 '12 - Resultado Financeiro Líquido',
 '14 - (+/-) Outras Receitas e Despesas não Operacionais',
 '14 - Outras Receitas e Despesas não Operacionais',
 '4.1 - Custo da Mercadoria Vendida - CMV',
 '4.1 - Custo da Mercadoria Vendida CMV',
 '4.2 - Custo dos Serviços Prestados - CSP',
 '4.2 - Custo dos Serviços Prestados CSP',
 '8.6.5 - Multa - Veículos',
 '8.6.5 - Multa Veículos',
 '8.8.2.1 - Doações - Outras entidades',
 '8.8.2.1 - Doações Outras entidades']

In [60]:
# dicionário com os valores diferentes encontrados
linha_dict = {
  "11 - Resultado Operacional -Ebitda": "11 - Resultado Operacional Ebitda",
  "12 - (+/-) Resultado Financeiro Líquido": "12 - Resultado Financeiro Líquido",
  "14 - (+/-) Outras Receitas e Despesas não Operacionais": "14 - Outras Receitas e Despesas não Operacionais",
  "4.1 - Custo da Mercadoria Vendida - CMV": "4.1 - Custo da Mercadoria Vendida CMV",
  "4.2 - Custo dos Serviços Prestados - CSP": "4.2 - Custo dos Serviços Prestados CSP",
  "8.6.5 - Multa - Veículos": "8.6.5 - Multa Veículos",
  "8.8.2.1 - Doações - Outras entidades": "8.8.2.1 - Doações Outras entidades"
}

# aplica o dicionário para padronizar os valores
staging_dre["LinhaEstrutura"] = staging_dre["LinhaEstrutura"].replace(linha_dict)

In [61]:
staging_dre.head()

,MesRef,LinhaEstrutura,Unidade,Valor
0,2023-03-01,1 - Receita Operacional Bruta,9999111 - ESCRITÓRIO,0.0
1,2023-03-01,1.1 - Vendas de mercadorias,9999111 - ESCRITÓRIO,0.0
2,2023-03-01,1.2 - Prestação de Serviços,9999111 - ESCRITÓRIO,0.0
3,2023-03-01,2 - Deduções da Receita Operacional Bruta,9999111 - ESCRITÓRIO,0.0
4,2023-03-01,2.1 - Impostos Incidentes sobre Vendas,9999111 - ESCRITÓRIO,0.0


In [62]:
# exporta a transformação
staging_dre.to_csv(url_base + "staging_dre.csv", index=False)

### Fato

In [63]:
# puxa os arquivos salvos anteriormente
staging_csv = pd.read_csv(url_base + "staging_dre.csv")
estrutura_csv = pd.read_csv(url_base + "dim_estrutura.csv")
unidade_csv = pd.read_csv(url_base + "dim_unidade.csv")

In [64]:
# ordena os dados
staging_csv = staging_csv.sort_values(by="LinhaEstrutura")
estrutura_csv = estrutura_csv.sort_values(by="LinhaEstrutura")
unidade_csv = unidade_csv.sort_values(by="NomeNaFonte")

In [65]:
# junta staging e estrutura
fato_dre = staging_csv.merge(estrutura_csv, on="LinhaEstrutura", how="left")

In [66]:
# filtra os dados por "Conta"
fato_dre = fato_dre[fato_dre["TipoLinha"] == "Conta"]

In [67]:
# seleciona as colunas
fato_dre = fato_dre[["MesRef", "Unidade", "Valor", "IdEstrutura"]]

In [68]:
# ordena os dados
fato_dre = fato_dre.sort_values(by="Unidade")

In [69]:
# junta fato e unidade
fato_dre = fato_dre.merge(unidade_csv, left_on="Unidade", right_on="NomeNaFonte", how="inner")

In [70]:
# seleciona as colunas
fato_dre = fato_dre[["IdEstrutura", "IdUnidade", "MesRef", "Valor"]]

In [71]:
# converte o campo de data
fato_dre['MesRef'] = pd.to_datetime(fato_dre['MesRef'], format='%Y-%m-%d')

In [72]:
fato_dre.head()

,IdEstrutura,IdUnidade,MesRef,Valor
0,27,1,2023-03-01,-145.90
1,161,1,2023-04-01,-8208.75
2,82,1,2023-06-01,0.00
3,82,1,2023-04-01,0.00
4,103,1,2023-03-01,-102.89


In [73]:
# exporta a transformação
fato_dre.to_csv(url_base + "fato_dre.csv", index=False)

### Banco de dados

In [74]:
# cria uma conexão com o banco de dados
conn = sqlite3.connect(url_base + "sqlite_database.db")

In [75]:
# lê os arquivos exportados anteriormente
dim_unidade = pd.read_csv(url_base + "dim_unidade.csv")
dim_estrutura = pd.read_csv(url_base + "dim_estrutura.csv")
staging_dre = pd.read_csv(url_base + "staging_dre.csv")
fato_dre = pd.read_csv(url_base + "fato_dre.csv")

In [76]:
# salva os arquivos lidos no formato sql
dim_unidade.to_sql("dim_unidade", conn, if_exists="replace", index=False)
dim_estrutura.to_sql("dim_estrutura", conn, if_exists="replace", index=False)
staging_dre.to_sql("staging_dre", conn, if_exists="replace", index=False)
fato_dre.to_sql("fato_dre", conn, if_exists="replace", index=False)

3968

In [77]:
# fecha a conexão com o banco de dados
conn.close()